In [ ]:
import os
import gc
import pandas as pd
import xarray as xr
import numpy as np
from pathlib import Path

Due to large size data set I run this code in Google CoLab to extract data from grided format to parquet file

In [ ]:
# USER SETTINGS - edit these
CITY_FILE = "/content/drive/MyDrive/Project Aurora/gadb_country_declatlon.csv"   # your city coordinates CSV
NETCDF_FILE = "/content/drive/MyDrive/Project Aurora/Complete_TMAX_Daily_LatLong1_2020.nc"# your netCDF file
OUTPUT_DIR = "/content/drive/MyDrive/Project Aurora" # output folder

In [ ]:
# output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
country = pd.read_csv(CITY_FILE)
country.columns = ["country", "lat", "lon"]   # rename columns

In [ ]:
# For vectorized selection
lats = xr.DataArray(country["lat"].values, dims="points")
lons = xr.DataArray(country["lon"].values, dims="points")

print(f"Processing NetCDF: {NETCDF_FILE}")
print(f"Number of city points: {len(country)}")

Processing NetCDF: /content/drive/MyDrive/Project Aurora/Complete_TMAX_Daily_LatLong1_2020.nc
Number of city points: 4187


In [ ]:
# Download the file if it's a URL
if NETCDF_FILE.startswith("http"):
    import gdown
    # Extract file ID from Google Drive link
    file_id = NETCDF_FILE.split("/d/")[1].split("/")[0]
    download_url = f"https://drive.google.com/uc?id={file_id}"
    local_file = os.path.join(OUTPUT_DIR, "temp_netcdf.nc")
    gdown.download(download_url, local_file, quiet=False)
    ds = xr.open_dataset(local_file)
else:
    ds = xr.open_dataset(NETCDF_FILE)

print("Dataset loaded. Variables:", list(ds.data_vars))

Dataset loaded. Variables: ['date_number', 'year', 'month', 'day', 'day_of_year', 'land_mask', 'temperature', 'climatology']


In [ ]:
# Extract variables
temp_anom = ds["temperature"]     # shape: (time, latitude, longitude)
clim = ds["climatology"]          # shape: (day_number, latitude, longitude)
doy = ds["day_of_year"]           # shape: (time,)  (1..365)

In [ ]:
# Create climatology expanded to time dimension to compute absolute temperatures
# day_number is 1..365 (or 0..364 depending on dataset).
# We use (doy.astype(int) - 1) to index climatology day_number (0-based)
clim_daily = clim.isel(day_number=(doy.astype(int) - 1))

In [ ]:
# Absolute temperature = climatology_for_that_day + anomaly
absolute_temp = clim_daily + temp_anom

In [ ]:
# Vectorized selection: get values at nearest grid points for all city coordinates
temp_country = temp_anom.sel(latitude=lats, longitude=lons, method="nearest")
abs_country  = absolute_temp.sel(latitude=lats, longitude=lons, method="nearest")

In [ ]:
# Convert to DataFrame
df_temp = temp_country.to_dataframe().reset_index()

In [ ]:
df_abs  = abs_country.to_dataframe(name="absolute_temperature").reset_index()

In [ ]:
# df_temp contains columns: time, points, latitude, longitude, temperature
# df_abs is same but for absolute temp; rename and merge safely by index
# Rename the temperature columns to avoid collision
df_temp = df_temp.rename(columns={"temperature":"temperature_anomaly"})
df_abs  = df_abs.rename(columns={"temperature":"absolute_temperature"})

In [ ]:
# Combine
# These two DataFrames should align exactly in rows and order; merge on ['time','points']
df_comb = pd.merge(df_temp, df_abs[["time","points","absolute_temperature"]],
                   on=["time","points"], how="left", validate="1:1")

In [ ]:
# Add original requested coordinates & metadata from cities list
df_comb["country"] = df_comb["points"].map(country["country"])
# Save original lat/lon requested (not the grid cell coordinates) for clarity
df_comb["requested_latitude"] = df_comb["points"].map(country["lat"])
df_comb["requested_longitude"] = df_comb["points"].map(country["lon"])


In [ ]:
# Keep only requested columns and reorder
df_comb = df_comb[[
    "time", "requested_longitude", "requested_latitude",
    "temperature_anomaly", "absolute_temperature"
]]

In [ ]:
# Rename to match your requested column names exactly:
df_comb = df_comb.rename(columns={
    "requested_longitude":"longitude",
    "requested_latitude":"latitude"
})

In [ ]:
# -------------------------------------------------------------
#Fix incorrect 'time' column in df_comb
# -------------------------------------------------------------
# Load the correct time information from the NetCDF fields
# (these come from your 'time_df' or directly from ds)
correct_dates = pd.to_datetime({
    "year": ds["year"].values.astype(int),
    "month": ds["month"].values.astype(int),
    "day": ds["day"].values.astype(int)
})

# Number of cities / points
n_points = len(country)

In [ ]:
# Repeat each daily timestamp for each city point
# Example: if we have 3652 days and  cities:
# correct time vector length will be 3652 * 4851 rows
df_comb["time"] = np.repeat(correct_dates.values, n_points)


In [ ]:
# Save parquet for this decade
#(you can change 'snappy' to other compression  e.g 'brotli' if desired itcompress more)
base = Path(NETCDF_FILE).stem
out_parquet = os.path.join(OUTPUT_DIR, f"{base}_cities.parquet")
print(f"Saving parquet: {out_parquet}  (this may take a moment)")
df_comb.to_parquet(out_parquet, compression="brotli", index=False)

Saving parquet: /content/drive/MyDrive/Project Aurora/Complete_TMAX_Daily_LatLong1_2020_cities.parquet  (this may take a moment)


## Build time index table for this decade (time_index sequential 0..N-1 within this decade) (not necessary)

In [ ]:
df_time = pd.DataFrame({
    "time": pd.to_datetime(ds["time"].values),
    "year": ds["year"].values,
    "month": ds["month"].values,
    "day": ds["day"].values
})
df_time  = df_time .reset_index(drop=True)
df_time ["time"] = df_time ["time"].dt.normalize() # Normalize to keep only date
df_time ["time_index"] = np.arange(len(df_time ))

In [ ]:
# Keep only requested columns and reorder
df_time = df_time[[
    "time_index", "year", "month", "day"
]]

In [ ]:
time_out = os.path.join(OUTPUT_DIR, f"{base}_time_index.parquet")
df_time.to_parquet(time_out, compression="snappy", index=False)

print("Saved time index:", time_out)

Saved time index: /content/drive/MyDrive/Project Aurora/Complete_TMAX_Daily_LatLong1_2020_time_index.parquet


In [ ]:
# Clean up
if 'ds' in locals() and ds is not None:
    ds.close()
    del ds
# Delete other variables if they exist
for var in ['temp_anom', 'clim', 'doy', 'clim_daily', 'absolute_temp', 'temp_country', 'abs_country']:
    if var in locals():
        del locals()[var]
gc.collect()

0

In [ ]:
print("Done. Output saved to:", out_parquet)

Done. Output saved to: /content/drive/MyDrive/Project Aurora/Complete_TMAX_Daily_LatLong1_2020_cities.parquet
